In [1]:
import psycopg2

try:
    conn = psycopg2.connect(
        host="localhost",      # endereço do seu servidor
        port=5432,             # porta padrão
        user="postgres",       # seu usuário
        password="24212522",  # sua senha
        dbname="auxulios_UFC"  # nome do seu banco
    )
    cursor = conn.cursor()
    print("Conexao realizada com sucesso!")

    # Teste: listar tabelas
    cursor.execute("SELECT tablename FROM pg_tables WHERE schemaname='public';")
    tables = cursor.fetchall()
    print("Tabelas:", tables)

except Exception as e:
    print("Erro na conexao:", e)


Conexao realizada com sucesso!
Tabelas: [('usuario',), ('estudante',), ('solicitacao_bolsa',), ('bolsa',)]


In [3]:
import psycopg2
import pandas as pd


In [8]:
!pip install sqlalchemy psycopg2-binary


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import psycopg2

try:
    conn = psycopg2.connect(
        host="localhost",      # endereço do seu servidor
        port=5432,             # porta padrão
        user="postgres",       # seu usuário
        password="24212522",  # sua senha
        dbname="auxulios_UFC"  # nome do seu banco
    )
    cursor = conn.cursor()
    print("Conexao realizada com sucesso!")

    # Teste: listar tabelas
    cursor.execute("SELECT tablename FROM pg_tables WHERE schemaname='public';")
    tables = cursor.fetchall()
    print("Tabelas:", tables)

except Exception as e:
    print("Erro na conexao:", e)


Conexao realizada com sucesso!
Tabelas: [('usuario',), ('estudante',), ('solicitacao_bolsa',), ('bolsa',)]


In [4]:
# -*- coding: utf-8 -*-
import os
import panel as pn
import pandas as pd
import sqlalchemy
from sqlalchemy import text
import hvplot.pandas

# Configurações de ambiente e extensões
os.environ["PGCLIENTENCODING"] = "LATIN1"
pn.extension('tabulator', notifications=True)

# ===================== Conexão com o Banco =====================
DB_USER = "postgres"
DB_PASS = "24212522"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "auxulios_UFC"

engine = sqlalchemy.create_engine(
    f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    connect_args={"options": "-c client_encoding=latin1"}
)

def carregar_tabela(nome_tabela):
    return pd.read_sql_table(nome_tabela, engine)

def executar_sql(sql, params=None):
    with engine.connect() as conn:
        conn.execute(text(sql), params or {})
        conn.commit()

# ===================== 1. LOGIN =====================
login_email = pn.widgets.TextInput(name="E-mail", placeholder="Digite seu e-mail")
login_senha = pn.widgets.PasswordInput(name="Senha", placeholder="Digite sua senha")
login_btn = pn.widgets.Button(name="Entrar", button_type="primary")
login_status = pn.pane.Alert("", visible=False)

def fazer_login(event):
    df = carregar_tabela("usuario")
    usuario = df[df['email'] == login_email.value]
    if usuario.empty:
        login_status.object = "⚠️ Usuário não encontrado. Redirecionando..."; login_status.visible = True
        pn.state.add_periodic_callback(lambda: setattr(tabs, 'active', 1), 2000, count=1)
        return
    if str(usuario['senha'].values[0]) == login_senha.value:
        perfil = usuario['perfil'].values[0]
        login_status.object = f"✅ Logado como **{perfil}**"; login_status.alert_type = "success"; login_status.visible = True
        configurar_permissoes(perfil)
        tabs.active = 3 if perfil == "ADMINISTRADOR" else 2
    else:
        login_status.object = "❌ Senha incorreta."; login_status.alert_type = "danger"; login_status.visible = True

def configurar_permissoes(perfil):
    is_admin = (perfil == "ADMINISTRADOR")
    sol_bol_tipo.disabled = not is_admin
    sol_bol_nome.disabled = not is_admin
    sol_grid.disabled = not is_admin
    btn_graf.visible = is_admin

login_btn.on_click(fazer_login)
tela_login = pn.Column(login_email, login_senha, login_btn, login_status, width=400)

# ===================== 2. REGISTRO (SENHA OCULTA) =====================
usuario_email = pn.widgets.TextInput(name="E-mail", width=250)
usuario_senha = pn.widgets.PasswordInput(name="Senha", width=250)
usuario_perfil = pn.widgets.Select(name="Perfil", options=["ESTUDANTE", "ADMINISTRADOR"], width=250)
usuario_status = pn.pane.Alert("", visible=False)

usuario_grid = pn.widgets.Tabulator(
    theme='materialize', pagination='local', page_size=5, 
    hidden_columns=['senha'],
    buttons={'excluir': '<i class="fa fa-trash"></i>'}
)

def carregar_usuario(event=None):
    usuario_grid.value = carregar_tabela("usuario")

def excluir_usuario(event):
    if event.column == 'excluir':
        id_user = usuario_grid.value.iloc[event.row]['id_usuario']
        executar_sql("DELETE FROM usuario WHERE id_usuario = :id", {"id": int(id_user)})
        pn.state.notifications.success("Usuário excluído!")
        carregar_usuario()

usuario_grid.on_click(excluir_usuario)

def salvar_usuario(event):
    try:
        executar_sql("INSERT INTO usuario (email, senha, perfil) VALUES (:email, :senha, :perfil)", 
                     {"email": usuario_email.value, "senha": usuario_senha.value, "perfil": usuario_perfil.value})
        usuario_status.object = "✅ Sucesso! Voltando para Login..."; usuario_status.alert_type = "success"; usuario_status.visible = True
        carregar_usuario()
        pn.state.add_periodic_callback(lambda: setattr(tabs, 'active', 0), 2000, count=1)
    except Exception as e:
        usuario_status.object = f"❌ Erro: {e}"; usuario_status.visible = True

btn_salvar_usuario = pn.widgets.Button(name="Registrar", button_type="success")
btn_salvar_usuario.on_click(salvar_usuario)

tela_usuario = pn.Column(usuario_email, usuario_senha, usuario_perfil, btn_salvar_usuario, usuario_status, usuario_grid)

# ===================== 3. ESTUDANTE (ID_USUARIO OCULTO) =====================
est_mat = pn.widgets.TextInput(name="Matrícula")
est_nome = pn.widgets.TextInput(name="Nome Completo")
est_nasc = pn.widgets.DatePicker(name="Nascimento")
est_email = pn.widgets.TextInput(name="E-mail")
est_tel = pn.widgets.TextInput(name="Telefone")
est_curso = pn.widgets.Select(name="Curso", options=["ENGENHARIA DE SOFTWARE", "DESIGN DIGITAL", "CIÊNCIAS DA COMPUTAÇÃO", "ENGENHARIA DE COMPUTAÇÃO", "REDE DE COMPUTADORES", "SISTEMA DE INFORMAÇÃO"])

opcoes_bolsas = ["BOLSA DE INICIAÇÃO ACADÊMICA", "BOLSA DE INCENTIVO AO DESPORTO"]
opcoes_auxilios = ["AUXILIO MORADIA", "AUXILIO INGRESSANTE", "AUXILIO EMERGENCIAL"]
est_bol_tipo = pn.widgets.Select(name="Tipo de Benefício", options=["BOLSA", "AUXILIO"])
est_bol_nome = pn.widgets.Select(name="Nome da Bolsa", options=opcoes_bolsas)

def atualizar_nomes_est(event):
    if est_bol_tipo.value == "BOLSA":
        est_bol_nome.name, est_bol_nome.options = "Nome da Bolsa", opcoes_bolsas
    else:
        est_bol_nome.name, est_bol_nome.options = "Nome do Auxílio", opcoes_auxilios

est_bol_tipo.param.watch(atualizar_nomes_est, 'value')

# AJUSTE: hidden_columns=['id_usuario'] retira o campo da visualização
est_grid = pn.widgets.Tabulator(
    theme='materialize', pagination='local', page_size=5, 
    hidden_columns=['id_usuario'],
    buttons={'excluir': '<i class="fa fa-trash"></i>'}
)

def carregar_estudante(event=None):
    est_grid.value = carregar_tabela("estudante")

def excluir_estudante(event):
    if event.column == 'excluir':
        mat = est_grid.value.iloc[event.row]['matricula']
        executar_sql("DELETE FROM estudante WHERE matricula = :m", {"m": mat})
        pn.state.notifications.success("Estudante excluído!")
        carregar_estudante()

est_grid.on_click(excluir_estudante)

def salvar_estudante_completo(event):
    df_users = carregar_tabela("usuario")
    user_match = df_users[df_users['email'] == est_email.value]
    
    id_u = None
    if not user_match.empty:
        id_u = int(user_match['id_usuario'].values[0])
    
    sql = """INSERT INTO estudante (matricula, nome_completo, data_nascimento, email, telefone, curso, id_usuario) 
             VALUES (:m, :n, :d, :e, :t, :c, :id_u)"""
    try:
        executar_sql(sql, {
            "m": est_mat.value, "n": est_nome.value, "d": est_nasc.value, 
            "e": est_email.value, "t": est_tel.value, "c": est_curso.value, "id_u": id_u
        })
        
        df_bolsas = carregar_tabela("bolsa")
        id_b = df_bolsas[df_bolsas['nome'] == est_bol_nome.value]['id_bolsa'].values[0]
        executar_sql("INSERT INTO solicitacao_bolsa (matricula_estudante, id_bolsa, status) VALUES (:m, :b, 'PENDENTE')", 
                     {"m": est_mat.value, "b": int(id_b)})
        
        carregar_estudante()
        tabs.active = 3 
    except Exception as e:
        pn.state.notifications.error(f"Erro: {e}")

btn_salvar_est = pn.widgets.Button(name="Salvar Cadastro", button_type="success")
btn_salvar_est.on_click(salvar_estudante_completo)

tela_estudante = pn.Column(pn.Row(est_mat, est_nome), pn.Row(est_nasc, est_email), pn.Row(est_tel, est_curso), 
                           pn.pane.Markdown("### 🎁 Escolha do Benefício"), pn.Row(est_bol_tipo, est_bol_nome), 
                           btn_salvar_est, est_grid, width=900)

# ===================== 4. SOLICITAÇÃO =====================
sol_bol_tipo = pn.widgets.Select(name="Tipo de Benefício", options=["BOLSA", "AUXILIO"], disabled=True)
sol_bol_nome = pn.widgets.Select(name="Nome do Benefício", options=opcoes_bolsas, disabled=True)
sol_grid = pn.widgets.Tabulator(
    theme='materialize', pagination='local', page_size=10,
    editors={'status': {'type': 'list', 'values': ['PENDENTE', 'APROVADA', 'REPROVADA']},
             'id_solicitacao': None, 'matricula': None, 'nome_completo': None, 'beneficio': None},
    buttons={'excluir': '<i class="fa fa-trash"></i>'}
)

def filtrar_estudantes_por_beneficio(event=None):
    sql = """SELECT s.id_solicitacao, e.matricula, e.nome_completo, b.nome as beneficio, s.status
             FROM estudante e JOIN solicitacao_bolsa s ON e.matricula = s.matricula_estudante
             JOIN bolsa b ON s.id_bolsa = b.id_bolsa WHERE b.nome = :nome"""
    sol_grid.value = pd.read_sql(text(sql), engine, params={"nome": sol_bol_nome.value})

def excluir_solicitacao(event):
    if event.column == 'excluir':
        id_sol = sol_grid.value.iloc[event.row]['id_solicitacao']
        executar_sql("DELETE FROM solicitacao_bolsa WHERE id_solicitacao = :id", {"id": int(id_sol)})
        pn.state.notifications.success("Solicitação excluída!")
        filtrar_estudantes_por_beneficio()

sol_grid.on_click(excluir_solicitacao)

def atualizar_status_banco(event):
    if event.column == 'status' and event.value:
        id_sol = sol_grid.value.iloc[event.row]['id_solicitacao']
        executar_sql("UPDATE solicitacao_bolsa SET status = :s WHERE id_solicitacao = :id", {"s": event.value, "id": int(id_sol)})
        pn.state.notifications.success("Status atualizado!")

sol_bol_nome.param.watch(filtrar_estudantes_por_beneficio, 'value')
sol_grid.on_edit(atualizar_status_banco)

def atualizar_nomes_sol(event):
    sol_bol_nome.options = opcoes_bolsas if sol_bol_tipo.value == "BOLSA" else opcoes_auxilios
sol_bol_tipo.param.watch(atualizar_nomes_sol, 'value')

tela_solicitacao = pn.Column(pn.Row(sol_bol_tipo, sol_bol_nome), sol_grid, width=900)

# ===================== 5. GRÁFICO =====================
grafico_painel = pn.pane.HoloViews()
def carregar_grafico(event=None):
    df = pd.read_sql("SELECT status, COUNT(*) AS total FROM solicitacao_bolsa GROUP BY status", engine)
    if not df.empty: 
        grafico_painel.object = df.hvplot.bar(x="status", y="total", color="status")

btn_graf = pn.widgets.Button(name="Gerar Gráfico", button_type="primary", visible=False)
btn_graf.on_click(carregar_grafico)

# ===================== TABS E CARGA INICIAL =====================
tabs = pn.Tabs(("Login 🔐", tela_login), ("Registro 👤", tela_usuario), ("Estudante 🎓", tela_estudante), 
               ("Solicitação 📝", tela_solicitacao), ("Gráfico 📊", pn.Column(btn_graf, grafico_painel)))

carregar_usuario(); carregar_estudante(); carregar_grafico(); filtrar_estudantes_por_beneficio()
pn.Column(tabs, align="center").servable()

NotificationArea()

Column(align='center')
    [0] Tabs
        [0] Column(width=400)
            [0] TextInput(name='E-mail', placeholder='Digite seu e-mail')
            [1] PasswordInput(name='Senha', placeholder='Digite sua senha')
            [2] Button(button_type='primary', name='Entrar')
            [3] Alert(str, sizing_mode='stretch_width', visible=False)
        [1] Column
            [0] TextInput(name='E-mail', width=250)
            [1] PasswordInput(name='Senha', width=250)
            [2] Select(name='Perfil', options=['ESTUDANTE', ...], value='ESTUDANTE', width=250)
            [3] Button(button_type='success', name='Registrar')
            [4] Alert(str, sizing_mode='stretch_width', visible=False)
            [5] Tabulator(buttons={'excluir': '<i c...}, hidden_columns=['senha'], page_size=5, pagination='local', theme='materialize', value=   id_usuario  ...)
        [2] Column(width=900)
            [0] Row
                [0] TextInput(name='Matrícula')
                [1] TextInput(name='Nome Completo')
            [1] Row
                [0] DatePicker(name='Nascimento')
                [1] TextInput(name='E-mail')
            [2] Row
                [0] TextInput(name='Telefone')
                [1] Select(name='Curso', options=['ENGENHARIA DE SOFTWARE',...], value='ENGENHARIA DE SOFTWARE')
            [3] Markdown(str)
            [4] Row
                [0] Select(name='Tipo de Benefício', options=['BOLSA', 'AUXILIO'], value='BOLSA')
                [1] Select(name='Nome da Bolsa', options=['BOLSA DE INICIAÇÃO ACADÊ...], value='BOLSA DE INICIAÇÃO A...)
            [5] Button(button_type='success', name='Salvar Cadastro')
            [6] Tabulator(buttons={'excluir': '<i c...}, hidden_columns=['id_usuario'], page_size=5, pagination='local', theme='materialize', value=  matricula   ...)
        [3] Column(width=900)
            [0] Row
                [0] Select(disabled=True, name='Tipo de Benefício', options=['BOLSA', 'AUXILIO'], value='BOLSA')
                [1] Select(disabled=True, name='Nome do Benefício', options=['BOLSA DE INICIAÇÃO ACADÊ...], value='BOLSA DE INICIAÇÃO A...)
            [1] Tabulator(buttons={'excluir': '<i c...}, editors={'status': {'type': 'list'...}, page_size=10, pagination='local', theme='materialize', value=Empty DataFrame
Columns: [...)
        [4] Column
            [0] Button(button_type='primary', name='Gerar Gráfico', visible=False)
            [1] HoloViews(Bars, height=300, sizing_mode='fixed', width=700)